# Ordinary Least Squares

In this notebook we are going to explore the Ordinary Least Squares (OLS) method for linear regression. We will use the `statsmodels` library to perform the regression and explore its basic functionalities. We will also compare the results with the `pyfixest` library which is a faster alternative to `statsmodels` for the case of high-dimensional fixed effects models. Finally, we will explore the performance of OLS with high-dimensional covariates.

This notebook is based on the [Getting Started with Statsmodels](https://www.statsmodels.org/stable/gettingstarted.html), [OLS Example with Statsmodels](https://www.statsmodels.org/stable/examples/notebooks/generated/ols.html), and [Getting Started with PyFixest](https://py-econometrics.github.io/pyfixest/quickstart.html) tutorials.


In [1]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Statistical libraries
import statsmodels.api as sm
import statsmodels.formula.api as smf

from IPython.display import display, Latex

## OLS Estimation

Artificial data:

In [2]:
rng = np.random.default_rng(42) # for reproducibility
nsample = 100
x = np.linspace(0, 10, 100)
X = np.column_stack((x, x ** 2))
beta = np.array([1, 0.1, 10])
e = rng.normal(size=nsample)

Our model needs an intercept so we add a column of 1s:

In [3]:
X = sm.add_constant(X)
y = np.dot(X, beta) + e

Fit and summary:

(Use the `cov_type` argument to set the covariance estimator to use)

In [4]:
model = sm.OLS(y, X)
res1 = model.fit(cov_type='HC1')
print(res1.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 8.187e+06
Date:                Sat, 05 Sep 2026   Prob (F-statistic):          2.97e-254
Time:                        08:14:13   Log-Likelihood:                -112.82
No. Observations:                 100   AIC:                             231.6
Df Residuals:                      97   BIC:                             239.5
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.8294      0.261      3.182      0.0

In [5]:
# Print out parameters, standard errors, and r-squared with 2 decimal places
print('Parameters:', res1.params.round(2))
print('Standard errors:', res1.bse.round(2))
print('R-squared:', res1.rsquared.round(2))

Parameters: [0.83 0.26 9.98]
Standard errors: [0.26 0.12 0.01]
R-squared: 1.0


## Overfitting

(Overfitting Example) Suppose $X \sim N\left(0, I_p\right)$ and $Y \sim N(0,1)$ are statistically independent. It follows that the best linear predictor of $Y$ is $\beta^{\prime} X = 0$ and that $R_{pop}^2=0$.

- If $p=n$, then the typical $R_{\text {sample }}^2$ is $1 \gg 0$.
- If $p=n / 2$, then the typical $R_{\text {sample }}^2$ is about $.5 \gg 0$.
- If $p=n / 20$, then the typical $R_{\text {sample }}^2$ is about $.05>0$.

These results can be deduced by simulation or analytically.

In [6]:
# Define a function producing measures of predictions
def regression_stats(n, p):
  # Set random seed for reproducibility
  np.random.seed(123)
  # Generate features and labels independently so that β0 = 0
  X = np.random.normal(size=(n, p))
  y = np.random.normal(size=(n, 1))
  # Run OLS regression
  model = sm.OLS(y, X).fit()
  # Print the ratio p/n
  print(f"p/n is: {p/n if n != 0 else np.inf:.2f}")
  # R2
  display(Latex(f"$R^2 = {model.rsquared:.3f}$"))
  # MSE
  display(Latex(f"$MSE  = {np.mean(model.resid**2):.3f}$"))
  # Adj_R2
  display(Latex(f"$AdjR^2  = {np.mean(model.rsquared_adj):.3f}$"))

In [7]:
# Let the simulation begin
regression_stats(1000, 50)
regression_stats(1000, 300)
regression_stats(1000, 500)
regression_stats(1000, 800)
regression_stats(1000, 999)

p/n is: 0.05


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

p/n is: 0.30


...

<IPython.core.display.Latex object>

p/n is: 1.00


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

---
## Additional material: OLS in high dimensions

These sections extend the examples above using `Econ524_Lab1_OLS_HighDim.ipynb`.
The main additions are out-of-sample prediction, rank and interpolation, confidence-interval
coverage, and a wage-data application. Run the original imports first, then these cells in order.
The simulations use separate random generators for reproducibility. The CPS section needs
internet access; the simulation sections do not.

**Counting parameters.** Below, $p$ counts all columns of the design matrix, including an
intercept if present. The Gaussian simulations have **no intercept**, matching the original
`regression_stats` example. In that example, statsmodels reports *uncentered* $R^2$,
$1-\mathrm{RSS}/\sum_i y_i^2$. The wage example includes an intercept and uses centered $R^2$.

### 1. Why training fit improves while prediction deteriorates

Let $y=Z\beta_0+u$, where $Z$ is $n\times p$, $E[u\mid Z]=0$, and
$\operatorname{Var}(u\mid Z)=\sigma^2 I_n$. For a full-column-rank design with $p<n$,
$\hat u=(I-P_Z)u$ and $\operatorname{tr}(I-P_Z)=n-p$. Therefore,

$$E[\mathrm{MSE}_{\mathrm{train}}\mid Z]=\sigma^2(1-p/n).$$

If the rows of $Z$ and a new test observation are independent $N(0,I_p)$ draws, then for
**$n>p+1$**,

$$E[\mathrm{MSE}_{\mathrm{test}}]
=\sigma^2\left(1+\frac{p}{n-p-1}\right).$$

The expectation averages over training samples and new observations. The second term is
estimation error; the first is irreducible noise. The exact denominator is specific to this
Gaussian design without an intercept. For small $p/n$, the excess prediction MSE is about
$\sigma^2p/n$, so root prediction error for the conditional mean has order $\sqrt{p/n}$.
Near $p=n$, that small-ratio approximation is inappropriate.

If the design has rank $r<p$, training MSE instead has expectation $\sigma^2(1-r/n)$
when the conditional mean lies in its column space. Do not apply the full-rank test formula
by simply substituting the number of columns of a redundant design.

In [ ]:
def hd_prediction_run(n, p, rng, sigma=1.0, n_test=2000, beta0=None):
    beta0 = np.zeros(p) if beta0 is None else np.asarray(beta0)
    Z = rng.standard_normal((n, p))
    y = Z @ beta0 + sigma * rng.standard_normal(n)
    bhat = np.linalg.lstsq(Z, y, rcond=None)[0]
    Z_test = rng.standard_normal((n_test, p))
    y_test = Z_test @ beta0 + sigma * rng.standard_normal(n_test)
    return np.mean((y - Z @ bhat)**2), np.mean((y_test - Z_test @ bhat)**2)

hd_rng = np.random.default_rng(524)
hd_n, hd_reps, hd_sigma = 120, 100, 1.0  # Increase repetitions for more precision.
hd_rows = []
for hd_p in [3, 12, 30, 60, 90, 108]:
    hd_draws = np.array([hd_prediction_run(hd_n, hd_p, hd_rng, hd_sigma)
                        for _ in range(hd_reps)])
    hd_rows.append({
        'p': hd_p, 'p/n': hd_p / hd_n,
        'Train MSE': hd_draws[:, 0].mean(),
        'Train theory': hd_sigma**2 * (1 - hd_p / hd_n),
        'Test MSE': hd_draws[:, 1].mean(),
        'Test theory': hd_sigma**2 * (1 + hd_p / (hd_n - hd_p - 1)),
        'Test MC SE': hd_draws[:, 1].std(ddof=1) / np.sqrt(hd_reps),
    })
hd_prediction_table = pd.DataFrame(hd_rows)
display(hd_prediction_table.round(3))

hd_fig, hd_ax = plt.subplots(figsize=(7, 4))
for observed, theoretical, marker in [('Train MSE', 'Train theory', 'o'),
                                       ('Test MSE', 'Test theory', 's')]:
    hd_ax.plot(hd_prediction_table['p/n'], hd_prediction_table[observed],
               marker=marker, label=observed)
    hd_ax.plot(hd_prediction_table['p/n'], hd_prediction_table[theoretical],
               linestyle='--', label=theoretical)
hd_ax.axhline(hd_sigma**2, color='gray', linestyle=':', label='Predict-zero benchmark')
hd_ax.set(xlabel='p / n', ylabel='MSE', title='Fitting noise: training versus test error')
hd_ax.legend()
hd_fig.tight_layout()
plt.show()

Every coefficient in this simulation is zero: the optimal predictor is zero, with population
MSE equal to 1. At $p/n=0.5$, test MSE is approximately 2 even though training MSE is about 0.5.
`Test MC SE` measures Monte Carlo uncertainty in the reported average. The theoretical curves
are expectations, so individual simulations need not be monotone.

**Exercise 1.** Repeat with `beta0=np.r_[np.ones(5), np.zeros(p-5)]` for $p\geq5$.
When all five relevant variables are always included, the expected OLS estimation error is
unchanged by their coefficient values. To study omitted-variable bias versus variance, instead
generate one fixed model with five nonzero coefficients and vary how many of its columns you
include in the fitted model. Keep the outcome-generating process fixed across specifications.

### 2. Interpolation and non-unique coefficients

At $p=n$, an invertible square design has the unique solution $Z^{-1}y$, which interpolates
the training outcomes. There are no residual degrees of freedom to estimate noise variance.
At $p>n$, a full-row-rank design still interpolates, but its null space has dimension $p-n$:
if $Zv=0$, then $\hat\beta+cv$ fits equally well for every scalar $c$.

`lstsq` chooses a minimum-Euclidean-norm solution when the coefficients are not unique.
That convention does not make all coefficients identified by the sample. Rank deficiency can
also occur at $p<n$, for example with duplicate columns or redundant dummy interactions.

In [ ]:
hd_rank_rng = np.random.default_rng(525)
hd_square = hd_rank_rng.standard_normal((40, 40))
hd_square_y = hd_rank_rng.standard_normal(40)
hd_square_b = np.linalg.solve(hd_square, hd_square_y)
print('p = n: training MSE =', np.mean((hd_square_y - hd_square @ hd_square_b)**2))

hd_wide = hd_rank_rng.standard_normal((40, 60))
hd_wide_y = hd_rank_rng.standard_normal(40)
hd_min_b, _, hd_rank, _ = np.linalg.lstsq(hd_wide, hd_wide_y, rcond=None)
_, _, hd_vt = np.linalg.svd(hd_wide, full_matrices=True)
hd_null_vector = hd_vt[hd_rank]
hd_alt_b = hd_min_b + 3.7 * hd_null_vector
print('p > n: rank =', hd_rank, '; null-space dimension =', hd_wide.shape[1] - hd_rank)
print('Minimum-norm residual norm:', np.linalg.norm(hd_wide_y - hd_wide @ hd_min_b))
print('Alternative residual norm:', np.linalg.norm(hd_wide_y - hd_wide @ hd_alt_b))
print('Coefficient distance:', np.linalg.norm(hd_alt_b - hd_min_b))
hd_new_Z = hd_rank_rng.standard_normal((2000, 60))
print('RMS difference in new predictions:',
      np.sqrt(np.mean((hd_new_Z @ (hd_alt_b - hd_min_b))**2)))

**Discussion.** Why can two models with identical training predictions disagree on new observations?
Distinguish sample rank deficiency from population identification: a Gaussian population can have
$E[Z_iZ_i']=I_p$ even when a particular sample has $p>n$. Population singularity is a separate
problem. For computation, use least-squares solvers instead of explicitly inverting $Z'Z$;
forming $Z'Z$ also worsens numerical conditioning.

### 3. Many controls and robust confidence intervals

The first OLS example uses HC1 standard errors. Here we check how often nominal 95% intervals
cover a known coefficient in repeated samples of
$y_i=\theta_0D_i+Z_i'\gamma_0+u_i$. Set $\gamma_0=0$ and let
$u_i=(0.5+|D_i|)\varepsilon_i$ to introduce heteroskedasticity.
The total column count $p$ includes $D$ and the $p-1$ controls; there is no intercept.

Write $H=X(X'X)^{-1}X'$ and $h_i=H_{ii}$; average leverage is $p/n$.
Under homoskedasticity, $E[\hat u_i^2\mid X]=\sigma^2(1-h_i)$.
Under heteroskedasticity the expression is instead
$E[\hat u_i^2\mid X]=\sum_j(I-H)_{ij}^2\sigma_j^2$.
Thus a simple degrees-of-freedom correction need not remove the bias.

| Estimator | Residual-square weight in the sandwich variance |
|---|---|
| HC0 | $\hat u_i^2$ |
| HC1 | $\hat u_i^2 n/(n-p)$ |
| HC2 | $\hat u_i^2/(1-h_i)$ |
| HC3 | $\hat u_i^2/(1-h_i)^2$ |

The code computes the variance of the first coefficient using a QR decomposition. It uses
normal critical values for all four methods, so the comparison isolates the covariance correction.

In [ ]:
def hd_coverage(n, p, reps=400, theta0=0.0, heteroskedastic=True, seed=526):
    if not 1 <= p < n:
        raise ValueError('Coverage experiment requires 1 <= p < n.')
    local_rng = np.random.default_rng(seed)
    hits = np.zeros(4)
    for _ in range(reps):
        Xc = local_rng.standard_normal((n, p))
        scale = 0.5 + np.abs(Xc[:, 0]) if heteroskedastic else np.ones(n)
        yc = theta0 * Xc[:, 0] + scale * local_rng.standard_normal(n)
        Q, R = np.linalg.qr(Xc, mode='reduced')
        bc = np.linalg.solve(R, Q.T @ yc)
        residual = yc - Xc @ bc
        leverage = np.sum(Q**2, axis=1)
        # First row of (X'X)^(-1)X' without explicitly forming an inverse.
        unit = np.zeros(p)
        unit[0] = 1
        influence = Q @ np.linalg.solve(R.T, unit)
        weights = np.column_stack([
            residual**2,
            residual**2 * n / (n - p),
            residual**2 / (1 - leverage),
            residual**2 / (1 - leverage)**2,
        ])
        standard_errors = np.sqrt((influence**2) @ weights)
        hits += np.abs(bc[0] - theta0) <= 1.96 * standard_errors
    return dict(zip(['HC0', 'HC1', 'HC2', 'HC3'], hits / reps))

hd_coverage_reps = 400
hd_coverage_table = pd.DataFrame([
    {'p/n': p / 200, **hd_coverage(200, p, reps=hd_coverage_reps)}
    for p in [4, 20, 60, 100]
])
display(hd_coverage_table.round(3))
print('Monte Carlo SE at 95% coverage:', round(np.sqrt(.95 * .05 / hd_coverage_reps), 3))

Compare each entry with 0.95 and allow for simulation uncertainty. HC0 often under-covers;
HC1 and HC2 can also under-cover with many controls and heteroskedasticity. HC3 can be
conservative, but none of these statements is a universal ordering across designs.
These ordinary robust estimators are not general solutions to inference with many covariates.

**Exercise 2.** Re-run with `heteroskedastic=False`. HC1 and HC2 should improve, but HC0 still
suffers residual shrinkage and HC3 can remain conservative. Under homoskedastic Gaussian errors,
the conventional OLS variance estimator with a $t_{n-p}$ critical value provides a useful benchmark.
Increase `reps` before interpreting differences of only one or two percentage points.

### 4. CPS wage data: flexibility versus validation performance

Use the same March 2015 CPS wage subsample as the reference notebook, available from the
[Applied Causal Inference data repository](https://raw.githubusercontent.com/CausalAIBook/MetricsMLNotebooks/main/data/wage2015_subsample_inference.csv).
The response is **log wage**. Compare basic controls, experience interactions, and a full set
of pairwise interactions on one fixed 50/50 split. These are predictive regressions; the
coefficient on `sex` is not automatically a causal effect.

Patsy supplies an intercept. Report both the total column count and training rank, and use rank
for residual degrees of freedom and adjusted training R-squared. Category levels are defined
on the full data only to keep a common feature schema; no held-out outcomes enter estimation.

In [ ]:
import patsy

hd_cps_url = ('https://raw.githubusercontent.com/CausalAIBook/MetricsMLNotebooks/'
              'main/data/wage2015_subsample_inference.csv')
hd_wages = pd.read_csv(hd_cps_url)
for hd_category in ['occ2', 'ind2']:
    hd_wages[hd_category] = hd_wages[hd_category].astype('category')
print('CPS data shape:', hd_wages.shape)
display(hd_wages[['lwage', 'sex', 'exp1', 'shs', 'hsg', 'scl', 'clg']].head())

hd_formulas = {
    'Basic': 'lwage ~ sex + exp1+exp2+exp3+exp4 + shs+hsg+scl+clg+occ2+ind2+mw+so+we',
    'Flexible': 'lwage ~ sex + (exp1+exp2+exp3+exp4)*(shs+hsg+scl+clg+occ2+ind2+mw+so+we)',
    'Extra flexible': 'lwage ~ sex + (shs+hsg+scl+clg+occ2+ind2+mw+so+we+exp1+exp2+exp3+exp4)**2',
}
hd_split = np.random.default_rng(7).permutation(len(hd_wages))
hd_train, hd_test = np.split(hd_split, [len(hd_wages)//2])

def hd_centered_r2(actual, predicted):
    return 1 - np.sum((actual - predicted)**2) / np.sum((actual - actual.mean())**2)

hd_wage_rows = []
hd_wage_designs = {}
for hd_name, hd_formula in hd_formulas.items():
    hd_response, hd_design = patsy.dmatrices(hd_formula, hd_wages, NA_action='raise')
    hd_yw, hd_Xw = np.asarray(hd_response).ravel(), np.asarray(hd_design)
    hd_bw, _, hd_training_rank, _ = np.linalg.lstsq(hd_Xw[hd_train], hd_yw[hd_train], rcond=None)
    hd_fit_train, hd_fit_test = hd_Xw[hd_train] @ hd_bw, hd_Xw[hd_test] @ hd_bw
    hd_train_r2 = hd_centered_r2(hd_yw[hd_train], hd_fit_train)
    hd_ntrain = len(hd_train)
    hd_wage_rows.append({
        'Model': hd_name, 'Columns (incl. intercept)': hd_Xw.shape[1],
        'Train rank': hd_training_rank, 'Columns/n_train': hd_Xw.shape[1] / hd_ntrain,
        'Rank/n_train': hd_training_rank / hd_ntrain,
        'Train MSE': np.mean((hd_yw[hd_train] - hd_fit_train)**2),
        'Train R2': hd_train_r2,
        'Adjusted train R2': (1 - (1 - hd_train_r2) * (hd_ntrain - 1) / (hd_ntrain - hd_training_rank)
                              if hd_training_rank < hd_ntrain else np.nan),
        'Test MSE': np.mean((hd_yw[hd_test] - hd_fit_test)**2),
        'Test R2': hd_centered_r2(hd_yw[hd_test], hd_fit_test),
        'Test MSE (train-mean baseline)': np.mean((hd_yw[hd_test] - hd_yw[hd_train].mean())**2),
    })
    hd_wage_designs[hd_name] = (hd_yw, hd_Xw)
hd_wage_table = pd.DataFrame(hd_wage_rows)
display(hd_wage_table.round(3))

**Read your computed results.** Which specification has the best training fit? Which has the
lowest test MSE? Do adjusted training $R^2$ and test performance rank the models the same way?
Do not assume the precise numbers or rankings reported in another notebook hold on every split.

A negative centered test $R^2$ means the predictions are worse than the constant **test-sample
mean of log wage**. That is an ex-post benchmark. The table separately reports the test MSE of
the **training-sample mean**, which is a constant predictor available without seeing test outcomes.

Compare the column count with training rank: interactions between mutually exclusive categories
can be zero or redundant. A solver can return coefficients despite this redundancy, but individual
coefficients need not be unique. The Gaussian test-error formula from Section 1 is not an exact
formula for this heterogeneous, potentially rank-deficient wage design.

**Exercise 3.** Change only the split seed and compare model rankings. If this held-out set is used
repeatedly to choose a model, it functions as a validation set. Reserve another untouched test set
for a final performance assessment.

### 5. Bridge to W2: validation and regularization

Validation measures generalization; regularization changes the estimator. With the normalization
below, LASSO solves

$$\min_{a,\beta}\ \frac{1}{2n}\sum_{i=1}^n(y_i-a-x_i'\beta)^2
+\lambda\sum_{j=1}^p|\beta_j|.$$

The intercept $a$ is not penalized. The penalty encourages sparse coefficients. Under suitable
sparsity, design, and noise assumptions, prediction-error rates can depend on
$\sqrt{s\log(p)/n}$ rather than $\sqrt{p/n}$, where $s$ is the number of relevant coefficients.
This is motivation for W2, not a guarantee for an arbitrary dataset.

The $\ell_1$ penalty is convex but **not strictly convex**, so LASSO coefficients can still be
non-unique with redundant predictors. Ridge with a positive $\ell_2$ penalty gives unique slope
coefficients. Neither penalty alone delivers valid conventional confidence intervals after selection.

**Exercise 4 (for W2).** Fit the extra-flexible wage specification with LASSO. Remove the Patsy
intercept column and let the estimator fit an unpenalized intercept. Use a `StandardScaler` and
`Lasso` pipeline inside `GridSearchCV` so each fold learns scaling only from its training rows;
choose the penalty by cross-validation within `hd_train`. Evaluate once on `hd_test` and report
MSE, $R^2$, and the number of nonzero slopes. Compare with the three OLS models without assuming
LASSO must win. If you tune further using these results, reserve a new final test set.

**Checkpoint.** Explain why (1) a high training $R^2$ need not imply useful prediction,
(2) perfect fit need not imply unique coefficients, and (3) heteroskedasticity-robust standard
errors need not give reliable coverage when the number of controls is large.